In [ ]:
# Notebook 全局导入与超参数。只改这一格，然后从上到下重新运行整个 notebook。
from pathlib import Path
import importlib.util
import sys
import warnings

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from skopt import BayesSearchCV
from skopt.space import Categorical, Integer, Real

warnings.filterwarnings("ignore")

support_path_candidates = [
    Path("cuda_training_support.py"),
    Path("..") / "cuda_training_support.py",
]
SUPPORT_PATH = next((path.resolve() for path in support_path_candidates if path.exists()), None)
if SUPPORT_PATH is None:
    raise FileNotFoundError("找不到 cuda_training_support.py")

support_spec = importlib.util.spec_from_file_location("cuda_training_support", SUPPORT_PATH)
if support_spec is None or support_spec.loader is None:
    raise ImportError(f"无法加载训练辅助模块: {SUPPORT_PATH}")

cuda_training_support = importlib.util.module_from_spec(support_spec)
sys.modules["cuda_training_support"] = cuda_training_support
support_spec.loader.exec_module(cuda_training_support)

# 如果内核从项目根目录启动，先移除本地 lightgbm 目录对官方包导入的遮蔽。
current_dir = Path.cwd().resolve()
if (current_dir / "lightgbm").is_dir():
    sys.path = [
        path
        for path in sys.path
        if Path(path or current_dir).resolve() != current_dir
    ]

import lightgbm as lgb

NOTEBOOK_RANDOM_SEED = 114514
NOTEBOOK_TEST_SIZE = 0.25
NOTEBOOK_BAYES_N_ITER = 24
NOTEBOOK_CV_FOLDS = 5
NOTEBOOK_MODEL_N_JOBS = 1
NOTEBOOK_SEARCH_N_JOBS = 1
NOTEBOOK_SMOTE_K_NEIGHBORS = 5
NOTEBOOK_BAYES_SCORING = "roc_auc"
NOTEBOOK_BAYES_VERBOSE = 1

NOTEBOOK_LGBM_SEARCH_SPACES = {
    "num_leaves": Integer(15, 63),
    "learning_rate": Real(5e-3, 5e-2, prior="log-uniform"),
    "n_estimators": Integer(80, 360),
    "max_depth": Categorical([3, 4, 5, 6, 7, 8, 10]),
    "subsample": Real(0.65, 0.9),
    "colsample_bytree": Real(0.5, 0.85),
    "min_child_samples": Integer(20, 120),
    "min_split_gain": Real(1e-3, 0.2, prior="log-uniform"),
    "reg_alpha": Real(1e-3, 5.0, prior="log-uniform"),
    "reg_lambda": Real(1e-2, 20.0, prior="log-uniform"),
}

NOTEBOOK_CONFIG = cuda_training_support.build_notebook_run_config(
    bayes_n_iter=NOTEBOOK_BAYES_N_ITER,
    cv_folds=NOTEBOOK_CV_FOLDS,
    random_seed=NOTEBOOK_RANDOM_SEED,
    test_size=NOTEBOOK_TEST_SIZE,
    model_n_jobs=NOTEBOOK_MODEL_N_JOBS,
    search_n_jobs=NOTEBOOK_SEARCH_N_JOBS,
    smote_k_neighbors=NOTEBOOK_SMOTE_K_NEIGHBORS,
)
NOTEBOOK_CV = StratifiedKFold(
    n_splits=NOTEBOOK_CONFIG.cv_folds,
    shuffle=True,
    random_state=NOTEBOOK_CONFIG.random_seed,
)
DEFAULT_TARGET_COLUMN = cuda_training_support.DEFAULT_TARGET_COLUMN
DATA_PATH = cuda_training_support.resolve_data_path(start_dir=Path.cwd())

print(
    cuda_training_support.format_notebook_run_summary(
        NOTEBOOK_CONFIG,
        data_path=DATA_PATH,
    )
)


In [ ]:
# 加载数据、预处理并执行完整训练。
random_seed = NOTEBOOK_CONFIG.random_seed

data = cuda_training_support.load_training_dataframe(
    data_path=DATA_PATH,
    random_seed=random_seed,
)
print(f"loaded_rows={len(data)}")
print("label_distribution=")
print(data[DEFAULT_TARGET_COLUMN].value_counts())

prepared = cuda_training_support.prepare_lightgbm_training_data(
    data,
    target_column=DEFAULT_TARGET_COLUMN,
    random_state=NOTEBOOK_CONFIG.random_seed,
    test_size=NOTEBOOK_CONFIG.test_size,
    smote_k_neighbors=NOTEBOOK_CONFIG.smote_k_neighbors,
)
X_train = prepared["X_train"]
X_test = prepared["X_test"]
y_train = prepared["y_train"]
y_test = prepared["y_test"]
scaler = prepared["scaler"]

print(f"train_shape={X_train.shape}")
print(f"test_shape={X_test.shape}")
print("resampled_train_distribution=")
print(pd.Series(y_train).value_counts())

lgbm_version = cuda_training_support.validate_lightgbm_cuda_build(
    random_state=NOTEBOOK_CONFIG.random_seed,
)
print(f"lightgbm_version={lgbm_version}")
print("cuda_preflight=ok")

lgb_model = cuda_training_support.build_lgbm_classifier(
    random_state=NOTEBOOK_CONFIG.random_seed,
    model_n_jobs=NOTEBOOK_CONFIG.model_n_jobs,
)

bayes_search = BayesSearchCV(
    estimator=lgb_model,
    search_spaces=NOTEBOOK_LGBM_SEARCH_SPACES,
    n_iter=NOTEBOOK_CONFIG.bayes_n_iter,
    cv=NOTEBOOK_CV,
    scoring=NOTEBOOK_BAYES_SCORING,
    n_jobs=NOTEBOOK_CONFIG.search_n_jobs,
    verbose=NOTEBOOK_BAYES_VERBOSE,
    random_state=NOTEBOOK_CONFIG.random_seed,
)

bayes_search.fit(X_train, y_train)
best_lgb = bayes_search.best_estimator_

y_test_pred = best_lgb.predict(X_test)
y_test_proba = best_lgb.predict_proba(X_test)[:, 1]

validation_metrics = {
    "AUC": roc_auc_score(y_test, y_test_proba),
    "Accuracy": accuracy_score(y_test, y_test_pred),
    "Balanced Accuracy": balanced_accuracy_score(y_test, y_test_pred),
    "Precision": precision_score(y_test, y_test_pred),
    "Recall": recall_score(y_test, y_test_pred),
    "F1": f1_score(y_test, y_test_pred),
}
tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred).ravel()
validation_metrics["Specificity"] = tn / (tn + fp)
validation_confusion_matrix = confusion_matrix(y_test, y_test_pred)


In [ ]:
data['RETENTION_TIME'] = pd.to_numeric(data['RETENTION_TIME'], errors='coerce').astype('float64')
print("转换前 RETENTION_TIME 的示例值：")
print(data['RETENTION_TIME'].head())

non_numeric = pd.to_numeric(data['RETENTION_TIME'], errors='coerce').isna()
print()
print("非数值数据数量：", non_numeric.sum())
print("非数值数据示例：")
print(data[non_numeric]['RETENTION_TIME'].unique())


In [ ]:
print("样本总数:", len(data))
print("标签分布:")
data[DEFAULT_TARGET_COLUMN].value_counts()


In [ ]:
print("过采样方法:", "notebook_smote")
print("过采样后的训练集形状:", X_train.shape)
print("测试集形状:", X_test.shape)
print("过采样后的标签分布:", pd.Series(y_train).value_counts())


## LightGBM（CUDA-only）

- 第一个单元格统一负责 import、路径修正、notebook 全局配置和数据路径解析。
- 第二个单元格执行完整的数据加载、特征预处理、CUDA 预检和一次 BayesSearchCV 训练。
- 训练设备固定为 `cuda`，禁止 CPU fallback。
- 为了进一步抑制过拟合，搜索空间额外加入了 `min_split_gain`、`reg_alpha`、`reg_lambda`，并收紧了 `num_leaves`、`max_depth`、`subsample`、`colsample_bytree`、`min_child_samples`。
- `subsample_freq=1` 已在辅助模块里固定开启，确保 bagging 参数真正生效。
- 官方当前不支持 Windows 上的 CUDA 版 LightGBM。需要把训练移动到 Linux 或 WSL2，并先执行：

```bash
pip uninstall -y lightgbm
pip install lightgbm --no-binary lightgbm --config-settings=cmake.define.USE_CUDA=ON
```


In [ ]:
print("LightGBM CUDA preflight:", lgbm_version)
print("最佳参数组合:", bayes_search.best_params_)
print("最佳交叉验证AUC:", bayes_search.best_score_)
for metric, value in validation_metrics.items():
    print(f"验证集 {metric}: {value:.6f}")
print("验证集混淆矩阵:")
print(validation_confusion_matrix)


In [ ]:
# 使用最佳参数训练模型
best_lgb = bayes_search.best_estimator_

# ============ 训练集评估 ============
y_train_pred = best_lgb.predict(X_train)
y_train_proba = best_lgb.predict_proba(X_train)[:, 1]

train_metrics = {
    'AUC': roc_auc_score(y_train, y_train_proba),
    'Accuracy': accuracy_score(y_train, y_train_pred),
    'Balanced Accuracy': balanced_accuracy_score(y_train, y_train_pred),
    'Precision': precision_score(y_train, y_train_pred),
    'Recall': recall_score(y_train, y_train_pred),
    'F1': f1_score(y_train, y_train_pred)
}

tn, fp, fn, tp = confusion_matrix(y_train, y_train_pred).ravel()
train_metrics['Specificity'] = tn / (tn + fp)

# ============ 测试集评估 ============
y_test_pred = best_lgb.predict(X_test)
y_test_proba = best_lgb.predict_proba(X_test)[:, 1]

test_metrics = {
    'AUC': roc_auc_score(y_test, y_test_proba),
    'Accuracy': accuracy_score(y_test, y_test_pred),
    'Balanced Accuracy': balanced_accuracy_score(y_test, y_test_pred),
    'Precision': precision_score(y_test, y_test_pred),
    'Recall': recall_score(y_test, y_test_pred),
    'F1': f1_score(y_test, y_test_pred)
}

tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred).ravel()
test_metrics['Specificity'] = tn / (tn + fp)

print()
print("=== 训练集性能 ===")
for metric, value in train_metrics.items():
    print(f"{metric:<18}: {value:.4f}")

print()
print("=== 测试集性能 ===")
for metric, value in test_metrics.items():
    print(f"{metric:<18}: {value:.4f}")

print()
print("=== 泛化差距（训练集 - 测试集） ===")
for metric in ["AUC", "Accuracy", "Balanced Accuracy", "Precision", "Recall", "F1", "Specificity"]:
    print(f"{metric:<18}: {train_metrics[metric] - test_metrics[metric]:+.4f}")

print()
print("测试集混淆矩阵:")
print(confusion_matrix(y_test, y_test_pred))

plt.figure(figsize=(10, 6))
lgb.plot_importance(best_lgb, max_num_features=20)
plt.tight_layout()
plt.show()
